<a href="https://colab.research.google.com/github/Hemang2001/M.Tech_Project_Predictive_Maintenance_Railway_Domain/blob/main/Model2_Wheel_Bearing_Fault/Predictive_Maintenance_of_Wheel_Bearing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Manual labeling for Bounding boxes

In [38]:
!wget https://engineering.case.edu/sites/default/files/188.mat
!wget https://engineering.case.edu/sites/default/files/3008.mat

--2026-06-04 17:58:45--  https://engineering.case.edu/sites/default/files/188.mat
Resolving engineering.case.edu (engineering.case.edu)... 129.22.104.251
Connecting to engineering.case.edu (engineering.case.edu)|129.22.104.251|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2931672 (2.8M)
Saving to: ‘188.mat’

188.mat             100%[===================>]   2.79M  12.3MB/s    in 0.2s    

2026-06-04 17:58:45 (12.3 MB/s) - ‘188.mat’ saved [2931672/2931672]

--2026-06-04 17:58:46--  https://engineering.case.edu/sites/default/files/3008.mat
Resolving engineering.case.edu (engineering.case.edu)... 129.22.104.251
Connecting to engineering.case.edu (engineering.case.edu)|129.22.104.251|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 968072 (945K)
Saving to: ‘3008.mat’

3008.mat            100%[===================>] 945.38K  5.10MB/s    in 0.2s    

2026-06-04 17:58:46 (5.10 MB/s) - ‘3008.mat’ saved [968072/968072]



In [39]:
import os

In [40]:
!ls

106.mat  107.mat  188.mat  3008.mat  sample_data


In [41]:
def process(file, label):
    data = scipy.io.loadmat(file)
    key = [k for k in data.keys() if 'DE_time' in k][0]
    signal = data[key].flatten()

    window_size = 2000
    segments = []

    for i in range(0, len(signal)-window_size, window_size):
        segments.append(signal[i:i+window_size])

    features = []
    for s in segments:
        features.append([np.mean(s), np.std(s), np.max(s), np.min(s)])

    X = np.array(features)
    y = np.full(len(X), label)

    return X, y

In [42]:
X1, y1 = process('188.mat', 0)   # healthy
X2, y2 = process('3008.mat', 2)  # fault

In [43]:
y2[:len(y2)//2] = 1   # early warning


In [44]:

X = np.vstack((X1, X2))
y = np.concatenate((y1, y2))


In [33]:
labels = np.ones(len(features))   # if faulty file
# use np.zeros(len(features)) for normal file

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y)

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [46]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9032258064516129


In [52]:
print(model.predict([X[120]]))

[2]


In [54]:
def alert_system(sample_id, sample_features):

    pred = model.predict([sample_features])[0]
    prob = model.predict_proba([sample_features])[0]

    confidence = max(prob)

    if pred == 0:
        print(f"[GREEN] ✅ ID:{sample_id} | NORMAL | Confidence: {confidence:.2f}")

    elif pred == 1:
        print(f"[YELLOW] ⚠️ ID:{sample_id} | EARLY WARNING | Risk Score: {confidence:.2f}")
        print("Action: Schedule maintenance soon")

    elif pred == 2:
        print(f"[RED] 🚨 ID:{sample_id} | HIGH FAILURE RISK | Risk Score: {confidence:.2f}")
        print("Action: IMMEDIATE inspection required")

In [55]:
for i in range(len(X)):
    alert_system(i, X[i])

[GREEN] ✅ ID:0 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:1 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:2 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:3 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:4 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:5 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:6 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:7 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:8 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:9 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:10 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:11 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:12 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:13 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:14 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:15 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:16 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:17 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:18 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:19 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:20 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:21 | NORMAL | Confidence: 1.00
[GREEN] ✅ ID:22 | NORMAL | Confidence: 1.0

In [70]:
!pip install gradio


In [71]:
import gradio as gr
import numpy as np

def predict(mean, std, max_val, min_val):
    X = np.array([[mean, std, max_val, min_val]])
    pred = model.predict(X)[0]

    if pred == 0:
        return "✅ NORMAL"
    elif pred == 1:
        return "⚠️ YELLOW ALERT (Early Warning)"
    else:
        return "🚨 RED ALERT (High Risk Failure)"

ui = gr.Interface(
    fn=predict,
    inputs=["number", "number", "number", "number"],
    outputs="text",
    title="Predictive Maintenance Alert System"
)

ui.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c9143609fdff59c90.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [74]:
X[91]

array([-1.29923170e-03,  2.00373611e+00,  7.56671240e+00, -7.22898540e+00])